# Neighborhood renovation investment analysis

**Original Question:** Where would buying a property and renovating give the best investment opportunity in the housing dataset?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_housing = pd.read_csv("data/df_housing.csv")

## Task 1: Establish a clean analysis sample and create renovation-related features needed to compare renovated and non-renovated properties by neighborhood.

**Acceptance Criteria:** We have a cleaned working DataFrame with one row per sale, no critical missingness in core fields, and new columns capturing renovation status, remodel age, property age at sale, and price per square foot, ready for neighborhood-level analysis.

### 1.1: Validate that the dataset grain, key temporal fields, and neighborhood distribution are as expected for this analysis.
_Output: print_

In [ ]:
import numpy as np

# Basic dataset shape and Id uniqueness
print("Dataset shape:", df_housing.shape)
print("Id unique:", df_housing["Id"].is_unique)
print("Duplicate Id count:", int(df_housing["Id"].duplicated().sum()))

# Summary for key temporal fields
print("YearBuilt summary:")
print(df_housing["YearBuilt"].describe())
print("YearRemodAdd summary:")
print(df_housing["YearRemodAdd"].describe())
print("YrSold summary:")
print(df_housing["YrSold"].describe())

# Temporal consistency checks
mask_remod_before_build = df_housing["YearRemodAdd"] < df_housing["YearBuilt"]
mask_negative_remodel_age = (df_housing["YrSold"] - df_housing["YearRemodAdd"]) < 0

print(
    "Rows where YearRemodAdd is earlier than YearBuilt:",
    int(mask_remod_before_build.sum()),
)
print(
    "Rows where YrSold - YearRemodAdd is negative:",
    int(mask_negative_remodel_age.sum()),
)

# Remodel age at sale inspection
df_housing_temporal = df_housing.copy()
df_housing_temporal["remodel_age_at_sale"] = (
    df_housing_temporal["YrSold"] - df_housing_temporal["YearRemodAdd"]
)
print("Remodel age at sale summary (YrSold - YearRemodAdd):")
print(df_housing_temporal["remodel_age_at_sale"].describe())
print("Remodel age at sale value counts (top 15):")
print(df_housing_temporal["remodel_age_at_sale"].value_counts().sort_index().head(15))

# Counts by year sold
print("Counts by year sold:")
print(df_housing["YrSold"].value_counts().sort_index())

# Top 10 neighborhoods by number of sales
print("Top 10 neighborhoods by number of sales:")
print(df_housing["Neighborhood"].value_counts().head(10))

# Print impossible temporal rows if any exist
if mask_remod_before_build.any() or mask_negative_remodel_age.any():
    print("Impossible temporal rows for inspection:")
    df_impossible_temporal = df_housing.loc[
        mask_remod_before_build | mask_negative_remodel_age,
        ["Id", "YearBuilt", "YearRemodAdd", "YrSold", "Neighborhood"],
    ].copy()
    print(df_impossible_temporal)
else:
    print("No impossible temporal rows found.")


### 1.2: Apply minimal, targeted cleaning required for robust analysis without over-modifying the dataset.
_Output: print_

In [ ]:
import pandas as pd

df_housing_clean = df_housing.copy()
df_housing_clean = df_housing_clean.loc[df_housing_clean["Electrical"].notna()].copy()

print("Rows before:", len(df_housing))
print("Rows after:", len(df_housing_clean))
print(
    "Remaining missing Electrical values:",
    int(df_housing_clean["Electrical"].isna().sum()),
)

add_to_workspace(df_housing_clean)


### 1.3: Create key engineered features to capture renovation status, timing, and normalized sale prices.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

df_housing_invest = df_housing_clean.copy()

df_housing_invest["property_age_at_sale"] = (
    df_housing_invest["YrSold"] - df_housing_invest["YearBuilt"]
)
df_housing_invest["remodel_age_at_sale"] = (
    df_housing_invest["YrSold"] - df_housing_invest["YearRemodAdd"]
)
df_housing_invest["was_ever_remodeled"] = (
    df_housing_invest["YearRemodAdd"] > df_housing_invest["YearBuilt"]
).astype(int)
df_housing_invest["is_recent_renovation"] = (
    df_housing_invest["remodel_age_at_sale"] <= 10
).astype(int)
df_housing_invest["is_non_recent_renovation"] = (
    1 - df_housing_invest["is_recent_renovation"]
).astype(int)
df_housing_invest["sale_price_per_sqft"] = df_housing_invest[
    "SalePrice"
] / df_housing_invest["GrLivArea"].replace(0, np.nan)

print("Property age at sale summary:")
print(df_housing_invest["property_age_at_sale"].describe())
print("Remodel age at sale summary:")
print(df_housing_invest["remodel_age_at_sale"].describe())
print("Sale price per square foot summary:")
print(df_housing_invest["sale_price_per_sqft"].describe())
print("Recent versus non-recent renovation counts:")
print(df_housing_invest[["is_recent_renovation", "is_non_recent_renovation"]].sum())
print("Year-by-year crosstab of recent renovation:")
print(
    pd.crosstab(df_housing_invest["YrSold"], df_housing_invest["is_recent_renovation"])
)

add_to_workspace(df_housing_invest)


## Task 2: Quantify and visualize how renovation premiums vary across neighborhoods and over time, while checking sample sizes and volatility.

**Acceptance Criteria:** We obtain neighborhood-level tables showing renovation premiums (absolute and percentage, both in raw price and price per square foot), sample sizes for renovated vs non-renovated homes, and volatility metrics across years, along with at least two visuals that reveal which neighborhoods have strong and stable premiums.

### 2.1: Understand how many renovated and non-renovated sales exist in each neighborhood to judge estimate reliability.
_Output: print_

In [ ]:
import pandas as pd

# Summarize neighborhood-level renovation counts and context metrics
df_neighborhood_reno_counts = (
    df_housing_invest.groupby("Neighborhood")
    .agg(
        recent_sales=("is_recent_renovation", "sum"),
        not_recent_sales=("is_non_recent_renovation", "sum"),
        total_sales=("Id", "count"),
        median_sale_price=("SalePrice", "median"),
        median_sale_price_per_sqft=("sale_price_per_sqft", "median"),
    )
    .reset_index()
)

df_neighborhood_reno_counts["has_sufficient_recent_and_not_recent"] = (
    (df_neighborhood_reno_counts["recent_sales"] >= 10)
    & (df_neighborhood_reno_counts["not_recent_sales"] >= 10)
).astype(int)

# Print summary sorted by recent-renovation count
print(
    df_neighborhood_reno_counts.sort_values(
        by="recent_sales", ascending=False
    ).to_string(index=False)
)

add_to_workspace(df_neighborhood_reno_counts)


### 2.2: Estimate renovation premiums in sale price and price per square foot for each neighborhood with adequate data.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

eligible_neighborhoods = df_neighborhood_reno_counts.loc[
    df_neighborhood_reno_counts["has_sufficient_recent_and_not_recent"] == 1,
    "Neighborhood",
].tolist()

df_neighborhood_eligible = df_housing_invest.loc[
    df_housing_invest["Neighborhood"].isin(eligible_neighborhoods)
].copy()

df_neighborhood_premium_long = (
    df_neighborhood_eligible.groupby(["Neighborhood", "is_recent_renovation"])
    .agg(
        sales_count=("Id", "count"),
        median_sale_price=("SalePrice", "median"),
        mean_sale_price=("SalePrice", "mean"),
        median_sale_price_per_sqft=("sale_price_per_sqft", "median"),
        mean_sale_price_per_sqft=("sale_price_per_sqft", "mean"),
    )
    .reset_index()
)

df_neighborhood_premium_long["renovation_group"] = df_neighborhood_premium_long[
    "is_recent_renovation"
].map(
    {
        1: "recent",
        0: "not_recent",
    }
)

df_neighborhood_premium_wide = df_neighborhood_premium_long.pivot(
    index="Neighborhood",
    columns="renovation_group",
    values=[
        "sales_count",
        "median_sale_price",
        "mean_sale_price",
        "median_sale_price_per_sqft",
        "mean_sale_price_per_sqft",
    ],
)

df_neighborhood_premium_wide.columns = [
    f"{metric}_{group}" for metric, group in df_neighborhood_premium_wide.columns
]
df_neighborhood_premiums = df_neighborhood_premium_wide.reset_index()

df_neighborhood_premiums["median_sale_price_abs_premium"] = (
    df_neighborhood_premiums["median_sale_price_recent"]
    - df_neighborhood_premiums["median_sale_price_not_recent"]
)
df_neighborhood_premiums["median_sale_price_pct_premium"] = (
    df_neighborhood_premiums["median_sale_price_abs_premium"]
    / df_neighborhood_premiums["median_sale_price_not_recent"]
    * 100
)
df_neighborhood_premiums["median_sale_price_per_sqft_abs_premium"] = (
    df_neighborhood_premiums["median_sale_price_per_sqft_recent"]
    - df_neighborhood_premiums["median_sale_price_per_sqft_not_recent"]
)
df_neighborhood_premiums["median_sale_price_per_sqft_pct_premium"] = (
    df_neighborhood_premiums["median_sale_price_per_sqft_abs_premium"]
    / df_neighborhood_premiums["median_sale_price_per_sqft_not_recent"]
    * 100
)
df_neighborhood_premiums["mean_sale_price_abs_premium"] = (
    df_neighborhood_premiums["mean_sale_price_recent"]
    - df_neighborhood_premiums["mean_sale_price_not_recent"]
)
df_neighborhood_premiums["mean_sale_price_pct_premium"] = (
    df_neighborhood_premiums["mean_sale_price_abs_premium"]
    / df_neighborhood_premiums["mean_sale_price_not_recent"]
    * 100
)
df_neighborhood_premiums["mean_sale_price_per_sqft_abs_premium"] = (
    df_neighborhood_premiums["mean_sale_price_per_sqft_recent"]
    - df_neighborhood_premiums["mean_sale_price_per_sqft_not_recent"]
)
df_neighborhood_premiums["mean_sale_price_per_sqft_pct_premium"] = (
    df_neighborhood_premiums["mean_sale_price_per_sqft_abs_premium"]
    / df_neighborhood_premiums["mean_sale_price_per_sqft_not_recent"]
    * 100
)

df_neighborhood_premiums = df_neighborhood_premiums.sort_values(
    by="median_sale_price_per_sqft_pct_premium", ascending=False
).reset_index(drop=True)

print(df_neighborhood_premiums.to_string(index=False))

add_to_workspace(df_neighborhood_premiums)


### 2.3: Communicate which neighborhoods have the strongest renovation premiums in a quickly interpretable visual format.
_Output: figure_

In [ ]:
import plotly.express as px

# Prepare plotting data from the existing neighborhood premiums table
if len(df_neighborhood_premiums) > 0:
    df_plot_neighborhood_premiums = df_neighborhood_premiums.sort_values(
        by="median_sale_price_per_sqft_pct_premium", ascending=False
    ).copy()
    df_plot_neighborhood_premiums["Neighborhood"] = pd.Categorical(
        df_plot_neighborhood_premiums["Neighborhood"],
        categories=df_plot_neighborhood_premiums["Neighborhood"].tolist(),
        ordered=True,
    )

    fig = px.bar(
        df_plot_neighborhood_premiums,
        x="Neighborhood",
        y="median_sale_price_per_sqft_pct_premium",
        color="median_sale_price_not_recent",
        color_continuous_scale="Blues",
        title="Neighborhoods with Adequate Samples: Renovation Premium per Sq Ft vs Entry Cost",
        labels={
            "Neighborhood": "Neighborhood",
            "median_sale_price_per_sqft_pct_premium": "Renovation Premium in Sale Price per Sq Ft (%)",
            "median_sale_price_not_recent": "Baseline Non-Renovated Median Sale Price (USD)",
        },
    )
    fig.update_traces(marker_line_color="black", marker_line_width=1)
    fig.update_layout(xaxis_tickangle=-45)
    fig.update_yaxes(tickformat=".1f")
    fig.show()
else:
    print("No neighborhood premium data available for plotting.")


### 2.4: Measure how renovation premiums vary year-to-year within each neighborhood to gauge stability vs volatility.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

eligible_neighborhoods = df_neighborhood_reno_counts.loc[
    df_neighborhood_reno_counts["has_sufficient_recent_and_not_recent"] == 1,
    "Neighborhood",
].tolist()

df_volatility_source = df_housing_invest.loc[
    df_housing_invest["Neighborhood"].isin(eligible_neighborhoods)
].copy()

# Yearly neighborhood-level medians for recent vs not-recent sales

df_neighborhood_year_group = (
    df_volatility_source.groupby(["Neighborhood", "YrSold", "is_recent_renovation"])
    .agg(
        sales_count=("Id", "count"),
        median_sale_price_per_sqft=("sale_price_per_sqft", "median"),
    )
    .reset_index()
)

df_neighborhood_year_pivot = df_neighborhood_year_group.pivot(
    index=["Neighborhood", "YrSold"],
    columns="is_recent_renovation",
    values="median_sale_price_per_sqft",
).reset_index()

# Keep only neighborhood-year combinations with both groups present
if 0 not in df_neighborhood_year_pivot.columns:
    df_neighborhood_year_pivot[0] = np.nan
if 1 not in df_neighborhood_year_pivot.columns:
    df_neighborhood_year_pivot[1] = np.nan

df_neighborhood_year_pivot["yearly_pct_premium"] = (
    (df_neighborhood_year_pivot[1] - df_neighborhood_year_pivot[0])
    / df_neighborhood_year_pivot[0]
    * 100
)

df_neighborhood_year_valid = df_neighborhood_year_pivot.loc[
    df_neighborhood_year_pivot[[0, 1]].notna().all(axis=1)
].copy()

df_neighborhood_premium_volatility = (
    df_neighborhood_year_valid.groupby("Neighborhood")
    .agg(
        mean_yearly_premium=("yearly_pct_premium", "mean"),
        std_yearly_premium=("yearly_pct_premium", "std"),
        min_yearly_premium=("yearly_pct_premium", "min"),
        max_yearly_premium=("yearly_pct_premium", "max"),
        valid_years=("YrSold", "nunique"),
    )
    .reset_index()
)

df_neighborhood_premium_volatility["range_yearly_premium"] = (
    df_neighborhood_premium_volatility["max_yearly_premium"]
    - df_neighborhood_premium_volatility["min_yearly_premium"]
)

df_neighborhood_premium_volatility = (
    df_neighborhood_premium_volatility[
        [
            "Neighborhood",
            "mean_yearly_premium",
            "std_yearly_premium",
            "range_yearly_premium",
            "min_yearly_premium",
            "max_yearly_premium",
            "valid_years",
        ]
    ]
    .sort_values(by="std_yearly_premium", ascending=True)
    .reset_index(drop=True)
)

print(df_neighborhood_premium_volatility.to_string(index=False))

add_to_workspace(df_neighborhood_premium_volatility)


### 2.5: Show how renovation premiums evolve over time in the most promising neighborhoods.
_Output: figure_

In [ ]:
import pandas as pd
import plotly.express as px

# Select neighborhoods with below-median volatility and strong mean yearly premium
if len(df_neighborhood_premium_volatility) > 0:
    df_volatility_selection = df_neighborhood_premium_volatility.copy()
    volatility_median = df_volatility_selection["std_yearly_premium"].median()

    df_volatility_selection = df_volatility_selection.loc[
        df_volatility_selection["std_yearly_premium"] <= volatility_median
    ].copy()
    df_volatility_selection = (
        df_volatility_selection.sort_values(
            by=["mean_yearly_premium", "std_yearly_premium"], ascending=[False, True]
        )
        .head(5)
        .copy()
    )

    selected_neighborhoods = df_volatility_selection["Neighborhood"].tolist()

    df_selected_source = df_housing_invest.loc[
        df_housing_invest["Neighborhood"].isin(selected_neighborhoods)
    ].copy()

    df_selected_year_group = (
        df_selected_source.groupby(["Neighborhood", "YrSold", "is_recent_renovation"])
        .agg(median_sale_price_per_sqft=("sale_price_per_sqft", "median"))
        .reset_index()
    )

    df_selected_year_pivot = df_selected_year_group.pivot(
        index=["Neighborhood", "YrSold"],
        columns="is_recent_renovation",
        values="median_sale_price_per_sqft",
    ).reset_index()

    if 0 not in df_selected_year_pivot.columns:
        df_selected_year_pivot[0] = pd.NA
    if 1 not in df_selected_year_pivot.columns:
        df_selected_year_pivot[1] = pd.NA

    df_selected_year_pivot["yearly_pct_premium"] = (
        (df_selected_year_pivot[1] - df_selected_year_pivot[0])
        / df_selected_year_pivot[0]
        * 100
    )

    df_selected_year_valid = df_selected_year_pivot.loc[
        df_selected_year_pivot[[0, 1]].notna().all(axis=1)
    ].copy()

    df_selected_year_valid = df_selected_year_valid.sort_values(
        by=["Neighborhood", "YrSold"]
    ).copy()

    print(
        "Selected neighborhoods with below-median volatility and strong mean yearly premium:"
    )
    print(
        df_volatility_selection[
            ["Neighborhood", "mean_yearly_premium", "std_yearly_premium", "valid_years"]
        ].to_string(index=False)
    )
    print("Yearly premium data used for plotting:")
    print(
        df_selected_year_valid[
            ["Neighborhood", "YrSold", "yearly_pct_premium"]
        ].to_string(index=False)
    )

    fig = px.line(
        df_selected_year_valid,
        x="YrSold",
        y="yearly_pct_premium",
        color="Neighborhood",
        markers=True,
        title="Top Candidate Neighborhoods with Relatively Stable Renovation Premiums in Sale Price per Sq Ft",
        labels={
            "YrSold": "Year Sold",
            "yearly_pct_premium": "Yearly Renovation Premium (%)",
            "Neighborhood": "Neighborhood",
        },
    )
    fig.update_traces(mode="lines+markers")
    fig.update_yaxes(tickformat=".1f")
    fig.show()
else:
    print("No volatility summary available for selection.")


## Task 3: Integrate renovation premium, affordability, sample size, and volatility metrics to pinpoint neighborhoods that offer the best buy-renovate-sell investment opportunities.

**Acceptance Criteria:** A ranked table and narrative clearly identify which neighborhoods look most attractive for renovation investment given strong and stable premiums, reasonable entry prices, and sufficient transaction volume, while flagging neighborhoods that are risky or data-poor.

### 3.1: Combine premium, affordability, and volatility metrics into a practical ranking of neighborhoods.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

df_neighborhood_invest_scores = (
    pd.merge(
        df_neighborhood_premiums,
        df_neighborhood_premium_volatility,
        on="Neighborhood",
        how="inner",
    )
    .merge(
        df_neighborhood_reno_counts[
            [
                "Neighborhood",
                "recent_sales",
                "not_recent_sales",
                "total_sales",
                "has_sufficient_recent_and_not_recent",
            ]
        ],
        on="Neighborhood",
        how="left",
    )
    .reset_index(drop=True)
)

# Composite score: higher premium is better, lower volatility and lower entry price are better
premium_rank = df_neighborhood_invest_scores[
    "median_sale_price_per_sqft_pct_premium"
].rank(ascending=False, method="min")
volatility_rank = df_neighborhood_invest_scores["std_yearly_premium"].rank(
    ascending=True, method="min"
)
entry_price_rank = df_neighborhood_invest_scores["median_sale_price_not_recent"].rank(
    ascending=True, method="min"
)

# Normalize ranks to 0-1 and combine with weights
n_rows = len(df_neighborhood_invest_scores)
if n_rows > 1:
    df_neighborhood_invest_scores["premium_rank_score"] = 1 - (premium_rank - 1) / (
        n_rows - 1
    )
    df_neighborhood_invest_scores["volatility_rank_score"] = 1 - (
        volatility_rank - 1
    ) / (n_rows - 1)
    df_neighborhood_invest_scores["entry_price_rank_score"] = 1 - (
        entry_price_rank - 1
    ) / (n_rows - 1)
else:
    df_neighborhood_invest_scores["premium_rank_score"] = 1.0
    df_neighborhood_invest_scores["volatility_rank_score"] = 1.0
    df_neighborhood_invest_scores["entry_price_rank_score"] = 1.0

df_neighborhood_invest_scores["composite_score"] = (
    0.5 * df_neighborhood_invest_scores["premium_rank_score"]
    + 0.3 * df_neighborhood_invest_scores["volatility_rank_score"]
    + 0.2 * df_neighborhood_invest_scores["entry_price_rank_score"]
)

df_neighborhood_invest_scores = df_neighborhood_invest_scores.sort_values(
    by=["composite_score", "median_sale_price_per_sqft_pct_premium"],
    ascending=[False, False],
).reset_index(drop=True)

print("Top neighborhoods by composite investment score:")
print(
    df_neighborhood_invest_scores[
        [
            "Neighborhood",
            "composite_score",
            "median_sale_price_not_recent",
            "median_sale_price_per_sqft_pct_premium",
            "median_sale_price_pct_premium",
            "mean_yearly_premium",
            "std_yearly_premium",
            "recent_sales",
            "not_recent_sales",
            "total_sales",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

add_to_workspace(df_neighborhood_invest_scores)


### 3.2: Investigate whether apparent top neighborhoods are genuinely attractive renovation plays or simply high-end markets where all homes are expensive.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Identify the top 3 ranked neighborhoods from the saved scorecard
if "df_neighborhood_invest_scores" not in globals():
    raise NameError(
        "df_neighborhood_invest_scores is required but not found in the namespace."
    )

if "df_housing_invest" not in globals():
    raise NameError("df_housing_invest is required but not found in the namespace.")

# Use the existing ranked table directly; no recomputation
if len(df_neighborhood_invest_scores) == 0:
    print("No neighborhood investment scores available.")
else:
    df_top3_neighborhoods = df_neighborhood_invest_scores.head(3)[
        ["Neighborhood"]
    ].copy()
    top3_neighborhoods = df_top3_neighborhoods["Neighborhood"].tolist()

    df_housing_compare = df_housing_invest.copy()
    df_housing_compare["group"] = np.where(
        df_housing_compare["Neighborhood"].isin(top3_neighborhoods),
        "Top 3 neighborhoods",
        "All other neighborhoods",
    )

    # Non-renovated homes only for the stock comparison requested
    df_non_renovated_compare = df_housing_compare.loc[
        df_housing_compare["is_recent_renovation"] == 0
    ].copy()

    # Summary table for non-renovated homes: median, IQR, and share of recent renovations in each group
    df_non_renovated_summary = (
        df_non_renovated_compare.groupby("group")
        .agg(
            sales_count=("Id", "count"),
            recent_renovation_share=("is_recent_renovation", "mean"),
            OverallQual_median=("OverallQual", "median"),
            OverallQual_q1=("OverallQual", lambda s: s.quantile(0.25)),
            OverallQual_q3=("OverallQual", lambda s: s.quantile(0.75)),
            GrLivArea_median=("GrLivArea", "median"),
            GrLivArea_q1=("GrLivArea", lambda s: s.quantile(0.25)),
            GrLivArea_q3=("GrLivArea", lambda s: s.quantile(0.75)),
            property_age_at_sale_median=("property_age_at_sale", "median"),
            property_age_at_sale_q1=(
                "property_age_at_sale",
                lambda s: s.quantile(0.25),
            ),
            property_age_at_sale_q3=(
                "property_age_at_sale",
                lambda s: s.quantile(0.75),
            ),
            sale_price_per_sqft_median=("sale_price_per_sqft", "median"),
            sale_price_per_sqft_q1=("sale_price_per_sqft", lambda s: s.quantile(0.25)),
            sale_price_per_sqft_q3=("sale_price_per_sqft", lambda s: s.quantile(0.75)),
        )
        .reset_index()
    )

    # Convert IQR bounds into explicit IQR values for compact reading
    for col_base in [
        "OverallQual",
        "GrLivArea",
        "property_age_at_sale",
        "sale_price_per_sqft",
    ]:
        df_non_renovated_summary[f"{col_base}_iqr"] = (
            df_non_renovated_summary[f"{col_base}_q3"]
            - df_non_renovated_summary[f"{col_base}_q1"]
        )

    # Keep the table compact and easy to compare
    df_non_renovated_summary = (
        df_non_renovated_summary[
            [
                "group",
                "sales_count",
                "recent_renovation_share",
                "OverallQual_median",
                "OverallQual_iqr",
                "GrLivArea_median",
                "GrLivArea_iqr",
                "property_age_at_sale_median",
                "property_age_at_sale_iqr",
                "sale_price_per_sqft_median",
                "sale_price_per_sqft_iqr",
            ]
        ]
        .sort_values("group")
        .reset_index(drop=True)
    )

    # Also provide the same compact comparison for all sales to show whether the top neighborhoods skew higher-end overall
    df_all_sales_summary = (
        df_housing_compare.groupby("group")
        .agg(
            sales_count=("Id", "count"),
            recent_renovation_share=("is_recent_renovation", "mean"),
            OverallQual_median=("OverallQual", "median"),
            OverallQual_q1=("OverallQual", lambda s: s.quantile(0.25)),
            OverallQual_q3=("OverallQual", lambda s: s.quantile(0.75)),
            GrLivArea_median=("GrLivArea", "median"),
            GrLivArea_q1=("GrLivArea", lambda s: s.quantile(0.25)),
            GrLivArea_q3=("GrLivArea", lambda s: s.quantile(0.75)),
            property_age_at_sale_median=("property_age_at_sale", "median"),
            property_age_at_sale_q1=(
                "property_age_at_sale",
                lambda s: s.quantile(0.25),
            ),
            property_age_at_sale_q3=(
                "property_age_at_sale",
                lambda s: s.quantile(0.75),
            ),
            sale_price_per_sqft_median=("sale_price_per_sqft", "median"),
            sale_price_per_sqft_q1=("sale_price_per_sqft", lambda s: s.quantile(0.25)),
            sale_price_per_sqft_q3=("sale_price_per_sqft", lambda s: s.quantile(0.75)),
        )
        .reset_index()
    )

    for col_base in [
        "OverallQual",
        "GrLivArea",
        "property_age_at_sale",
        "sale_price_per_sqft",
    ]:
        df_all_sales_summary[f"{col_base}_iqr"] = (
            df_all_sales_summary[f"{col_base}_q3"]
            - df_all_sales_summary[f"{col_base}_q1"]
        )

    df_all_sales_summary = (
        df_all_sales_summary[
            [
                "group",
                "sales_count",
                "recent_renovation_share",
                "OverallQual_median",
                "OverallQual_iqr",
                "GrLivArea_median",
                "GrLivArea_iqr",
                "property_age_at_sale_median",
                "property_age_at_sale_iqr",
                "sale_price_per_sqft_median",
                "sale_price_per_sqft_iqr",
            ]
        ]
        .sort_values("group")
        .reset_index(drop=True)
    )

    # Format percentages for readability
    df_non_renovated_summary["recent_renovation_share"] = (
        df_non_renovated_summary["recent_renovation_share"] * 100
    ).round(1)
    df_all_sales_summary["recent_renovation_share"] = (
        df_all_sales_summary["recent_renovation_share"] * 100
    ).round(1)

    print("Non-renovated homes only: median and IQR by group")
    print(df_non_renovated_summary.to_string(index=False))
    print("")
    print("All sales: median and IQR by group")
    print(df_all_sales_summary.to_string(index=False))
    print("")
    print("Top 3 ranked neighborhoods:")
    print(", ".join(top3_neighborhoods))


### 3.3: Identify neighborhoods where renovation investment appears risky due to volatility or sparse data.
_Output: print_

In [ ]:
import pandas as pd

# Recreate the risk summary from the saved sample-count and volatility tables
if len(df_neighborhood_reno_counts) == 0:
    print("No neighborhood sample-count data available.")
else:
    df_risk_sample = df_neighborhood_reno_counts[
        [
            "Neighborhood",
            "recent_sales",
            "not_recent_sales",
            "total_sales",
            "has_sufficient_recent_and_not_recent",
        ]
    ].copy()

    df_risk_vol = df_neighborhood_premium_volatility[
        [
            "Neighborhood",
            "mean_yearly_premium",
            "std_yearly_premium",
            "range_yearly_premium",
            "min_yearly_premium",
            "max_yearly_premium",
            "valid_years",
        ]
    ].copy()

    df_risk_summary = pd.merge(
        df_risk_sample, df_risk_vol, on="Neighborhood", how="left"
    ).reset_index(drop=True)

    df_risk_summary["insufficient_recent_samples"] = (
        df_risk_summary["recent_sales"] < 10
    )
    df_risk_summary["insufficient_non_recent_samples"] = (
        df_risk_summary["not_recent_sales"] < 10
    )

    volatility_75th = df_risk_summary["std_yearly_premium"].quantile(0.75)
    df_risk_summary["high_volatility"] = (
        df_risk_summary["std_yearly_premium"] > volatility_75th
    )

    def build_reason_flags(row):
        reasons = []
        if row["insufficient_recent_samples"]:
            reasons.append("insufficient_recent_samples")
        if row["insufficient_non_recent_samples"]:
            reasons.append("insufficient_non_recent_samples")
        if row["high_volatility"]:
            reasons.append("high_volatility")
        return ", ".join(reasons) if reasons else ""

    df_risk_summary["caution_flags"] = df_risk_summary.apply(build_reason_flags, axis=1)

    df_caution_table = df_risk_summary.loc[
        df_risk_summary["caution_flags"] != ""
    ].copy()
    df_caution_table = df_caution_table.sort_values(
        by=["high_volatility", "std_yearly_premium", "Neighborhood"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    df_caution_table = df_caution_table[
        [
            "Neighborhood",
            "caution_flags",
            "recent_sales",
            "not_recent_sales",
            "total_sales",
            "std_yearly_premium",
            "mean_yearly_premium",
            "range_yearly_premium",
            "valid_years",
        ]
    ]

    print("Risk summary for neighborhoods")
    print(
        f"High volatility threshold (75th percentile of std yearly premium): {volatility_75th:.3f}"
    )
    print(df_caution_table.to_string(index=False))


### 3.4: Cross-check key findings for internal consistency and synthesize a direct answer to where buying and renovating offers the best opportunity.
_Output: print_

In [ ]:
import pandas as pd

# Recreate the risk summary from the saved sample-count and volatility tables
if len(df_neighborhood_reno_counts) == 0:
    print("No neighborhood sample-count data available.")
else:
    df_risk_sample = df_neighborhood_reno_counts[
        [
            "Neighborhood",
            "recent_sales",
            "not_recent_sales",
            "total_sales",
            "has_sufficient_recent_and_not_recent",
        ]
    ].copy()

    df_risk_vol = df_neighborhood_premium_volatility[
        [
            "Neighborhood",
            "mean_yearly_premium",
            "std_yearly_premium",
            "range_yearly_premium",
            "min_yearly_premium",
            "max_yearly_premium",
            "valid_years",
        ]
    ].copy()

    df_risk_summary = pd.merge(
        df_risk_sample, df_risk_vol, on="Neighborhood", how="left"
    ).reset_index(drop=True)

    df_risk_summary["insufficient_recent_samples"] = (
        df_risk_summary["recent_sales"] < 10
    )
    df_risk_summary["insufficient_non_recent_samples"] = (
        df_risk_summary["not_recent_sales"] < 10
    )

    volatility_75th = df_risk_summary["std_yearly_premium"].quantile(0.75)
    df_risk_summary["high_volatility"] = (
        df_risk_summary["std_yearly_premium"] > volatility_75th
    )

    def build_reason_flags(row):
        reasons = []
        if row["insufficient_recent_samples"]:
            reasons.append("insufficient_recent_samples")
        if row["insufficient_non_recent_samples"]:
            reasons.append("insufficient_non_recent_samples")
        if row["high_volatility"]:
            reasons.append("high_volatility")
        return ", ".join(reasons) if reasons else ""

    df_risk_summary["caution_flags"] = df_risk_summary.apply(build_reason_flags, axis=1)

    df_caution_table = df_risk_summary.loc[
        df_risk_summary["caution_flags"] != ""
    ].copy()
    df_caution_table = df_caution_table.sort_values(
        by=["high_volatility", "std_yearly_premium", "Neighborhood"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    df_caution_table = df_caution_table[
        [
            "Neighborhood",
            "caution_flags",
            "recent_sales",
            "not_recent_sales",
            "total_sales",
            "std_yearly_premium",
            "mean_yearly_premium",
            "range_yearly_premium",
            "valid_years",
        ]
    ]

    print("Risk summary for neighborhoods")
    print(
        f"High volatility threshold (75th percentile of std yearly premium): {volatility_75th:.3f}"
    )
    print(df_caution_table.to_string(index=False))
